# Module 4: Replication & Fault Tolerance

本章目标：
- 理解 Kafka 副本（Replica）机制如何保障数据持久性
- 区分 Leader Replica 与 Follower Replica 的职责
- 观察 ISR（In-Sync Replicas）的变化
- 模拟 Broker 宕机，观察 Leader 重新选举过程
- 理解 `acks` 参数对可靠性与延迟的权衡

---

## 前置条件

- 集群运行中：`docker compose up -d`（3 个 Broker）
- 已安装依赖：`uv sync`

## 核心概念

```
Topic: reliable-topic (replication_factor=3)

Partition 0:
  Broker 1 ← Leader   (接受读写)
  Broker 2 ← Follower (同步 Leader 数据)
  Broker 3 ← Follower (同步 Leader 数据)

ISR = {1, 2, 3}  ← 与 Leader 保持同步的 Broker 集合

若 Broker 1 宕机：
  → 从 ISR 中选出新 Leader（Broker 2 或 3）
  → 数据不丢失（已同步到 ISR）
```

### acks 参数

| acks | 等待确认对象 | 丢失风险 | 延迟 |
|------|------------|---------|------|
| 0 | 不等待 | 最高 | 最低 |
| 1 | Leader 写入即返回 | 中等（Leader 宕机可能丢失）| 低 |
| all / -1 | 所有 ISR 写入完成 | 最低 | 较高 |

In [ ]:
import asyncio
from aiokafka import AIOKafkaProducer
from aiokafka.admin import AIOKafkaAdminClient, NewTopic

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TOPIC = "reliable-topic"

## 1. 创建高可靠 Topic（RF=3, min.insync.replicas=2）

In [ ]:
async def create_reliable_topic(brokers, topic):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if topic in existing:
            print(f"Topic '{topic}' already exists")
        else:
            await admin.create_topics([
                NewTopic(
                    name=topic,
                    num_partitions=3,
                    replication_factor=3,
                    topic_configs={
                        "min.insync.replicas": "2",  # 至少 2 个 ISR 成员才允许写入
                    }
                )
            ])
            print(f"✓ Topic '{topic}' created (RF=3, min.insync.replicas=2)")

        # 显示副本状态
        descriptions = await admin.describe_topics([topic])
        for desc in descriptions:
            print(f"\nTopic: {desc['topic']}")
            for p in sorted(desc['partitions'], key=lambda x: x['partition']):
                print(
                    f"  P{p['partition']}: leader={p['leader']}, "
                    f"replicas={p['replicas']}, isr={p['isr']}"
                )
    finally:
        await admin.close()

await create_reliable_topic(BROKERS, TOPIC)

## 2. 对比不同 acks 设置的行为

In [ ]:
import time

async def benchmark_acks(brokers, topic, acks_value, label, count=20):
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        acks=acks_value,
    )
    await producer.start()
    start = time.perf_counter()
    try:
        for i in range(count):
            await producer.send_and_wait(topic, f"acks={acks_value} msg-{i}".encode())
    finally:
        await producer.stop()
    elapsed = time.perf_counter() - start
    print(f"  {label:20s} acks={str(acks_value):<4} | {count} msgs | {elapsed*1000:.1f} ms total | {elapsed*1000/count:.1f} ms/msg")

print("=== acks benchmark ===")
for acks, label in [(0, "fire-and-forget"), (1, "leader-ack"), ("all", "all-isr-ack")]:
    await benchmark_acks(BROKERS, TOPIC, acks, label)

## 3. 模拟 Broker 宕机：Leader 重新选举

> **注意**：以下步骤需要在终端手动执行 Docker 命令。
> 本 Notebook 负责监测集群状态变化。

**操作步骤：**
1. 运行下面的「持续监测」单元格
2. 在另一个终端执行：`docker compose stop kafka-1`
3. 观察 Leader 变化和 ISR 收缩
4. 恢复：`docker compose start kafka-1`
5. 观察 Broker 重新加入 ISR

In [ ]:
async def monitor_topic_state(brokers, topic, interval=3, rounds=10):
    """每隔 interval 秒打印一次 Topic 的副本状态。"""
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        for r in range(rounds):
            print(f"\n--- Round {r+1} ---")
            try:
                descriptions = await admin.describe_topics([topic])
                for desc in descriptions:
                    for p in sorted(desc['partitions'], key=lambda x: x['partition']):
                        isr_str = str(sorted(p['isr']))
                        alert = " ⚠️  ISR degraded!" if len(p['isr']) < 3 else ""
                        print(
                            f"  P{p['partition']}: leader={p['leader']:2d} | "
                            f"replicas={sorted(p['replicas'])} | isr={isr_str}{alert}"
                        )
            except Exception as e:
                print(f"  Error: {e}")
            await asyncio.sleep(interval)
    finally:
        await admin.close()

# 运行 10 轮（30秒），期间可在终端 stop/start kafka-1
print("监测中...在另一个终端执行 'docker compose stop kafka-1'")
print("30 秒后自动停止（中断 Kernel 可提前结束）\n")
await monitor_topic_state(BROKERS, TOPIC, interval=3, rounds=10)

## 4. 幂等 Producer（防止网络重试导致消息重复）

In [ ]:
async def idempotent_producer_demo(brokers, topic):
    # enable_idempotence=True 会自动设置 acks='all', retries=MAX_INT
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        enable_idempotence=True,
    )
    await producer.start()
    try:
        for i in range(5):
            meta = await producer.send_and_wait(
                topic, f"idempotent-{i}".encode()
            )
            print(f"  sent: idempotent-{i} -> partition={meta.partition}, offset={meta.offset}")
    finally:
        await producer.stop()
    print("幂等 Producer 保证：即使重试，每条消息仅被写入一次")

await idempotent_producer_demo(BROKERS, TOPIC)

## 总结

| 概念 | 要点 |
|------|------|
| Replication Factor | 每个 Partition 的副本数；RF=3 允许 1 个 Broker 失败 |
| Leader | 负责读写的主副本；Follower 被动同步 |
| ISR | 与 Leader 保持同步的副本集；`min.insync.replicas` 控制最低要求 |
| acks=all | 等待所有 ISR 确认，最强持久性保证 |
| 幂等 Producer | 防止重试导致消息重复写入 |

## 下一章

**Module 5: KRaft Architecture** — 深入了解 Kafka 3.x 的 KRaft 共识机制，告别 ZooKeeper。